# Personalized FL for Brain-Tumor Segmentation — Colab runner

Runs the full pipeline (centralized ceiling → local floor → FedAvg/FedProx/**FedBN**/fine-tune → analyze) on a **Colab** GPU, so it doesn't tie up your own machine. The synthetic per-hospital scanner shift is ON, so FedBN has heterogeneity to specialize on (hypotheses H1–H3).

**Before you run:**
1. `Runtime → Change runtime type → GPU` (T4 is fine).
2. Run the cells top to bottom. Each `!command` runs on the Colab VM.
3. You provide the BraTS data in Cell 4 (Kaggle token or Drive).

> **Honest note on speed:** this pipeline is **CPU-bound** (nibabel slice reads + the scanner-shift transforms), not GPU-bound. Colab's value here is *offloading your PC*, not necessarily raw speed — free Colab often gives only ~2 vCPUs, which can be **slower** than a many-core desktop. Colab Pro / high-RAM runtimes give more cores. The run cell auto-detects and uses whatever cores Colab allocates.

## Cell 1 — check GPU, Python, and cores

In [ ]:
import sys, os, subprocess
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                    capture_output=True, text=True).stdout or "NO GPU — set Runtime > Change runtime type > GPU")
print("Python", sys.version.split()[0], "| CPU cores:", os.cpu_count())

## Cell 2 — clone the repo (latest `master`)

In [ ]:
%cd /content
!rm -rf capstone
!git clone --depth 1 https://github.com/Ravijangid820/capstone.git
%cd /content/capstone
!git log --oneline -1

## Cell 3 — install dependencies

Colab already ships CUDA-enabled **torch**, so we install everything *except* torch (reinstalling it can break Colab's CUDA build). If `nvflare` prints a protobuf/grpcio conflict, re-run the cell once — it usually settles; if it hard-errors, send me the output.

In [ ]:
!pip install -q "monai>=1.3" "nibabel>=5.2" einops "nvflare>=2.5" \
    "scipy>=1.12" "scikit-image>=0.22" "scikit-learn>=1.4" "pandas>=2.1" \
    "matplotlib>=3.8" "tqdm>=4.66"
import torch, nvflare
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| nvflare", nvflare.__version__)

## Cell 4 — get the BraTS 2021 data  — **pick ONE option**

The ~15 GB isn't in the repo. Run **A** or **B**, then the verify cell.

### Option A — Kaggle
kaggle.com → Account → **Create New API Token** → upload the downloaded `kaggle.json` to `/content` (Files panel), then run:

In [ ]:
# --- OPTION A: Kaggle ---
!mkdir -p ~/.kaggle && cp /content/kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!pip install -q kaggle
!mkdir -p /content/brats
# Verify this dataset slug on kaggle.com first; swap if you use a different BraTS 2021 mirror.
!kaggle datasets download -d dschettler8845/brats-2021-task1 -p /content/brats
!cd /content/brats && (unzip -o -q '*.zip' || true)
!mkdir -p /content/capstone/data
!find /content/brats -name 'BraTS2021_Training_Data.tar' -exec tar -xf {} -C /content/capstone/data \;

### Option B — from Google Drive (if you already have BraTS there)
Uncomment and edit the paths.

In [ ]:
# --- OPTION B: Google Drive ---
# from google.colab import drive; drive.mount('/content/drive')
# !mkdir -p /content/capstone/data
# # if it's a tar:
# !tar -xf '/content/drive/MyDrive/PATH/BraTS2021_Training_Data.tar' -C /content/capstone/data
# # or a folder of case dirs:
# !cp -r '/content/drive/MyDrive/PATH/BraTS2021_Training_Data' /content/capstone/data/
print('Option B commented out — uncomment + edit paths if using Drive.')

### Verify data — auto-locates the case folders, sets the data root

In [ ]:
import glob, os
segs = glob.glob('/content/**/*_seg.nii.gz', recursive=True)
assert segs, "No *_seg.nii.gz found — the data download/extraction didn't produce case folders. Check Cell 4."
case_dirs = sorted(set(os.path.dirname(s) for s in segs))
DR = os.path.commonpath(case_dirs) if len(case_dirs) > 1 else os.path.dirname(case_dirs[0])
os.environ['DR'] = DR
print(f"found {len(case_dirs)} cases | data root = {DR}")
print("example case files:", sorted(os.listdir(case_dirs[0]))[:6])

## Cell 5 — run the pipeline

Scoped defaults (`MAX_CASES=200 ROUNDS=20 EPOCHS=20`) finish in a couple of hours and are enough to see whether **FedBN beats FedAvg on the shifted hospitals**. `WORKERS` auto-uses all Colab cores. Colab has no `uv`, so we patch `run_all.sh` to call plain `python` (with the repo's `src/` on `PYTHONPATH`).

**For the full headline run:** set `MAX_CASES=''` (all cases) and raise `ROUNDS=50 EPOCHS=40` — but keep it inside one Colab session (no mid-run resume yet).

In [ ]:
import os
os.environ['PYTHONPATH'] = '/content/capstone/src'
os.environ['GPU']       = '0'
os.environ['MAX_CASES'] = '200'      # '' = all cases (full run)
os.environ['ROUNDS']    = '20'       # 50 for the full run
os.environ['EPOCHS']    = '20'       # 40 for the full run
os.environ['BATCH']     = '8'
os.environ['WORKERS']   = str(os.cpu_count() or 4)
print('WORKERS =', os.environ['WORKERS'], '| MAX_CASES =', os.environ['MAX_CASES'] or 'ALL')
%cd /content/capstone
# Colab has no uv -> make run_all.sh use plain python
!sed -i 's|uv run python|python|g' run_all.sh
!bash run_all.sh

## Cell 6 — the H1–H3 result

Want: **FedBN ≥ FedAvg on the shifted hospitals** (site-4/5/6) and FedBN ≥ FedAvg on average. Positive bars in `personalization_gain.png` = the core result (H3).

In [ ]:
import pandas as pd
from IPython.display import Image, display
print('=== per-hospital mean Dice ===')
print(pd.read_csv('/content/capstone/results/per_hospital.csv').to_string(index=False))
print('\n=== summary (avg / worst-hospital / spread) ===')
print(pd.read_csv('/content/capstone/results/summary.csv').to_string(index=False))
display(Image('/content/capstone/results/personalization_gain.png'))
display(Image('/content/capstone/results/comparison.png'))

## Cell 7 — save results to Drive (so they survive the session)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/capstone_results
!cp -r /content/capstone/results /content/drive/MyDrive/capstone_results/
!cp /content/capstone/data/centralized_unet.pt /content/drive/MyDrive/capstone_results/ 2>/dev/null || true
print('saved results + centralized checkpoint to Drive/MyDrive/capstone_results')